In [1]:
import sqlite3
from pathlib import Path
from collections import defaultdict, OrderedDict
from typing import Dict, Iterable, Optional, Tuple

APP_NAME_PLAIN = OrderedDict([
    ("A1", "WhatsApp"),
    ("A2", "Snapchat"),
    ("A3", "Telegram"),
    ("A4", "Google Maps"),
    ("A5", "Samsung Internet"),
    ("I1", "WhatsApp"),
    ("I2", "Contacts"),
    ("I3", "Apple Messages"),
    ("I4", "Safari"),
    ("I5", "Calendar"),
])

PATTERNS = ("*.db", "*.sqlite", "*.sqlitedb", "*.sqlite3")


# -------------------------
# Utilities
# -------------------------

def get_app_code_from_filename(db_file: Path) -> str:
    stem = db_file.stem
    if "_" in stem:
        return stem.split("_", 1)[0]
    if "-" in stem:
        return stem.split("-", 1)[0]
    return stem


def count_columns_in_db(db_path: Path) -> int:
    """
    Counts only real physical tables.
    Excludes:
      - sqlite_* internal tables
      - VIRTUAL TABLE definitions (prevents tokenizer errors)
    """
    conn: Optional[sqlite3.Connection] = None
    total_cols = 0

    try:
        conn = sqlite3.connect(str(db_path))
        cur = conn.cursor()

        # Only physical tables, skip virtual tables entirely
        cur.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
              AND name NOT LIKE 'sqlite_%'
              AND sql NOT LIKE '%VIRTUAL TABLE%';
        """)

        tables = [row[0] for row in cur.fetchall()]

        for table_name in tables:
            try:
                cur.execute(f'PRAGMA table_info("{table_name}");')
                cols = cur.fetchall()
                total_cols += len(cols)
            except sqlite3.Error:
                # Skip problematic tables safely
                continue

    except sqlite3.Error as e:
        raise RuntimeError(f"{db_path}: {e}")

    finally:
        if conn:
            conn.close()

    return total_cols


def iter_db_files(in_dir: Path, patterns: Iterable[str]) -> Iterable[Path]:
    seen = set()
    for pat in patterns:
        for fp in in_dir.glob(pat):
            p = fp.resolve()
            if p in seen:
                continue
            seen.add(p)
            yield fp


# -------------------------
# Main CSV writer
# -------------------------

def write_app_column_totals(
    in_dir: str | Path,
    out_csv: str | Path,
    patterns: Tuple[str, ...] = PATTERNS,
) -> Path:

    in_dir = Path(in_dir)
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    if not in_dir.exists():
        raise FileNotFoundError(f"Input folder not found: {in_dir.resolve()}")

    totals_by_app: Dict[str, int] = defaultdict(int)

    files = list(iter_db_files(in_dir, patterns))

    if not files:
        out_csv.write_text("app_code,app_name,total_columns\n", encoding="utf-8")
        return out_csv

    for fp in sorted(files):
        app_code = get_app_code_from_filename(fp)
        col_count = count_columns_in_db(fp)
        totals_by_app[app_code] += col_count

    # Deterministic ordering
    app_order = list(APP_NAME_PLAIN.keys()) + [
        a for a in sorted(totals_by_app.keys()) if a not in APP_NAME_PLAIN
    ]

    lines = ["app_code,app_name,total_columns"]

    for app_code in app_order:
        if app_code not in totals_by_app:
            continue
        app_name = APP_NAME_PLAIN.get(app_code, app_code)
        lines.append(f"{app_code},{app_name},{totals_by_app[app_code]}")

    out_csv.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return out_csv


# -------------------------
# Runner
# -------------------------

if __name__ == "__main__":
    IN_DIR = Path(r"..\..\selectedDBs")
    OUT_CSV = Path("app_total_columns.csv")

    out = write_app_column_totals(IN_DIR, OUT_CSV, patterns=PATTERNS)
    print(f"Wrote: {out.resolve()}")


Wrote: I:\project2026\llmagent\RQs\RQ2\app_total_columns.csv
